# Conso_RTE_final.csv

In [2]:
import pandas as pd
from datetime import datetime, timedelta

# Fonction pour préparer un fichier
def prepare_yearly_file(filepath, start_date):
    df = pd.read_csv(filepath, sep=";", encoding="utf-8")

    # Renommer colonnes
    df.columns = ["Heures", "PrévisionJ-1", "PrévisionJ", "Consommation"] + list(df.columns[4:])

    # Nettoyage
    df = df[df["Heures"].notna()]
    df = df[df["Heures"].astype(str).str.strip() != ""]
    df = df[~df["Heures"].astype(str).str.startswith("Journée du")]
    df = df[df["Heures"].astype(str).str.lower() != "heures"]

    # Garder que les colonnes utiles
    df_clean = df[["Heures", "Consommation"]].copy().reset_index(drop=True)

    # Générer les datetimes réels pour cette année
    start_datetime = datetime.strptime(start_date, "%Y-%m-%d %H:%M")
    df_clean["datetime"] = [start_datetime + timedelta(minutes=15 * i) for i in range(len(df_clean))]

    return df_clean

# Traiter chaque fichier avec sa vraie date de départ
df_2023 = prepare_yearly_file("2023.csv", "2023-01-01 00:00")
df_2024 = prepare_yearly_file("2024.csv", "2024-01-01 00:00")
df_2025 = prepare_yearly_file("2025.csv", "2025-01-01 00:00")

# Fusionner tous les fichiers
df_all = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)

# Nettoyer la consommation
df_all.set_index("datetime", inplace=True)
df_all["Consommation"] = pd.to_numeric(df_all["Consommation"], errors="coerce")
df_all["Consommation"] = df_all["Consommation"].interpolate(method="time")

# Repasser en colonne
df_all.reset_index(inplace=True)
df_all["date"] = df_all["datetime"].dt.date
df_all["heure"] = df_all["datetime"].dt.strftime("%H:%M")
df_all["Consommation"] = df_all["Consommation"].round().astype(int)

# Couper au 24 février 2025 à 10h15
cutoff = datetime(2025, 2, 24, 10, 15)
df_all = df_all[df_all["datetime"] <= cutoff]

# Sauvegarde
df_final = df_all[["date", "heure", "Consommation"]]
df_final.to_csv("conso_RTE_final.csv", index=False)

# Aperçu
df_final.head()

,date,heure,Consommation
0,2023-01-01,00:00,46008
1,2023-01-01,00:15,46462
2,2023-01-01,00:30,45349
3,2023-01-01,00:45,44669
4,2023-01-01,01:00,45387


# Créer la colonne datetime

In [3]:
import pandas as pd

# Charger le fichier
df = pd.read_csv("conso_RTE_final.csv")

# Créer la colonne datetime à partir de date + heure
df["datetime"] = pd.to_datetime(df["date"] + " " + df["heure"], format="%Y-%m-%d %H:%M")

# Garder uniquement les colonnes utiles
df = df[["datetime", "Consommation"]]

# Sauvegarder dans un nouveau fichier
df.to_csv("conso_RTE_datetime_clean.csv", index=False)

# Aperçu du résultat
print(df.head())

             datetime  Consommation
0 2023-01-01 00:00:00         46008
1 2023-01-01 00:15:00         46462
2 2023-01-01 00:30:00         45349
3 2023-01-01 00:45:00         44669
4 2023-01-01 01:00:00         45387
